# Mastering Spark Job Concurrency: Python Multiprocessing Pool & Spark FIFO vs. FAIR Scheduling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

### Course Module: Advanced PySpark Optimization & Execution Internals
* **Author**: **K Sharath Rushi**
* **GitHub**: [github.com/sharathrushi](https://github.com/sharathrushi/)
* **LinkedIn**: [linkedin.com/in/ksharathrushik](https://www.linkedin.com/in/ksharathrushik/)
* **Domain**: Distributed Computing, Generative AI & Data Engineering
* **Target Environment**: Google Colab / Linux / Cloud Notebooks

---

### Executive Summary & Learning Objectives
A common assumption among data engineers is:
> *"If I wrap my PySpark actions in a Python worker pool (`Pool`), my Spark jobs will automatically execute concurrently on the cluster!"*

In reality, **Python-level concurrency is only half of the equation**. Apache Spark contains its own internal engine scheduler (`DAGScheduler` and `TaskScheduler`). By default, Spark operates in **FIFO (First-In, First-Out)** scheduling mode. When multiple jobs are submitted simultaneously from Python worker processes/threads, Spark queues them, giving 100% priority to the first job. A heavy batch job will completely starve a short interactive query, causing **head-of-line blocking**.

In this comprehensive hands-on guide, we will:
1. **Understand Python Concurrency**: Benchmark sequential vs. parallel execution using Python's `from multiprocessing import Pool`.
2. **Expose the Spark FIFO Bottleneck**: Dispatch two Spark jobs simultaneously via Python's `Pool` under Spark's default FIFO scheduler and observe the hidden sequential blocking.
3. **Unlock True Concurrency with FAIR Scheduling**: Reconfigure Spark to `spark.scheduler.mode = FAIR`, assign jobs to distinct scheduler pools, and prove how short jobs execute simultaneously alongside heavy jobs.
4. **Visualize the Execution Timelines**: Generate a high-resolution Matplotlib Gantt chart comparing the execution lifespans and latency across all three modes.
5. **Architectural Takeaways**: Learn how to configure production XML pools with weights and minimum share for multi-tenant and low-latency environments.


## 0. Environment Setup & Colab Initialization

Run the cell below to install **PySpark** and visualization dependencies in Google Colab. If you are already running this in an environment with PySpark installed, the command will quickly verify existing packages.


In [ ]:
# Step 0: Install PySpark and dependencies (Ready for Google Colab)
!pip install -q pyspark==3.5.1 matplotlib pandas tabulate

import os
import sys
import time
import multiprocessing
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt

print(f"Python Version    : {sys.version.split()[0]}")
print(f"Available CPU Cores: {multiprocessing.cpu_count()}")
print(f"Timestamp          : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


---
## 1. Python Sequential Processing vs. Parallel Processing (`multiprocessing.Pool`)

Before introducing Spark's distributed engine, let us ground the concept of concurrency in pure Python.

### Concept:
* **Sequential Execution**: Tasks run strictly one after another on a single core. Total time equals the sum of all task durations:
  $$\text{Total Time} = T_1 + T_2 + \dots + T_n$$
* **Parallel Execution (`multiprocessing.Pool`)**: Tasks are distributed across worker processes running simultaneously on multiple CPU cores. Total time is governed by the longest task plus process spawning overhead:
  $$\text{Total Time} \approx \max(T_1, T_2, \dots, T_n) + \text{overhead}$$


In [ ]:
# Define a standalone Python task simulating a discrete unit of work
def run_python_task(task_name, duration_sec):
    """
    Simulates a discrete unit of work and logs exact start/finish timestamps.
    """
    start_time = time.time()
    start_iso = datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"  [{start_iso}] [{task_name}] STARTED  (Expected Duration: {duration_sec}s)")
    
    # Simulate processing time
    time.sleep(duration_sec)
    
    end_time = time.time()
    end_iso = datetime.now().strftime("%H:%M:%S.%f")[:-3]
    actual_duration = end_time - start_time
    print(f"  [{end_iso}] [{task_name}] FINISHED (Actual Duration: {actual_duration:.2f}s)")
    
    return {
        "task_name": task_name,
        "start": start_time,
        "end": end_time,
        "duration": actual_duration
    }


In [ ]:
# --- 1A. Sequential Python Execution ---
print("======================================================================")
print(">>> 1A. RUNNING PYTHON TASKS SEQUENTIALLY")
print("======================================================================")

seq_start_wall = time.time()

# Run Task 1, then Task 2
res_seq_1 = run_python_task("Python-Task-1", duration_sec=3)
res_seq_2 = run_python_task("Python-Task-2", duration_sec=3)

seq_total_wall = time.time() - seq_start_wall

print(f"\nSequential Total Wall-Clock Time: {seq_total_wall:.2f}s (Theoretical: ~6.0s)\n")


In [ ]:
# --- 1B. Parallel Python Execution using multiprocessing.Pool ---
from multiprocessing import Pool

print("======================================================================")
print(">>> 1B. RUNNING PYTHON TASKS IN PARALLEL (multiprocessing.Pool)")
print("======================================================================")

tasks_to_run = [
    ("Python-Task-1", 3),
    ("Python-Task-2", 3)
]

par_start_wall = time.time()

# Spawn a pool with 2 worker processes
with Pool(processes=2) as pool:
    # starmap unpacks tuples as arguments to run_python_task
    par_results = pool.starmap(run_python_task, tasks_to_run)

par_total_wall = time.time() - par_start_wall
speedup = seq_total_wall / par_total_wall

print(f"\nParallel Total Wall-Clock Time  : {par_total_wall:.2f}s (Theoretical: ~3.0s)")
print(f"Calculated Python Speedup Factor: {speedup:.2f}x")


### Key Takeaway from Step 1:
In pure Python, wrapping tasks in `multiprocessing.Pool` creates independent worker processes that run concurrently across CPU cores, achieving a near **2x speedup**.

---
## 2. Bridging Python Concurrency to Apache Spark: The Two-Layer Architecture

Now, what happens when we apply this same pattern to trigger **Spark jobs**?

To answer this, we must understand the relationship between the **Python Driver** and the **Spark JVM Engine**:

```
+--------------------------------------------------------------------------+
|                        LAYER 1: PYTHON DRIVER                            |
|                                                                          |
|       Worker 1 (Job A)                     Worker 2 (Job B)              |
|              |                                    |                      |
|       apply_async()                        apply_async()                 |
|              v                                    v                      |
|   +------------------------------------------------------------------+   |
|   |         Py4J Gateway Client (Thread-Safe Socket Pool)           |   |
|   +------------------------------------------------------------------+   |
+-----------------------------------|--------------------------------------+
                                    |
                                    v
+-----------------------------------|--------------------------------------+
|                     LAYER 2: SPARK JVM ENGINE                            |
|                                                                          |
|                       SparkContext / SparkSession                        |
|                                   |                                      |
|                       DAGScheduler (Job -> Stages)                       |
|                                   |                                      |
|                 TaskScheduler (Stages -> Tasks -> Cores)                 |
|                                   |                                      |
|                  +----------------+----------------+                     |
|                  |                                 |                     |
|             FIFO Mode                         FAIR Mode                  |
|    (Job A consumes ALL cores;         (Tasks interleaved round-robin;    |
|     Job B queues behind Job A)        Job A & Job B run in parallel)     |
+--------------------------------------------------------------------------+
```

### Why `from multiprocessing.pool import ThreadPool as Pool` for Spark?
When coordinating Spark jobs from Python, the standard and officially recommended pattern is:
```python
from multiprocessing.pool import ThreadPool as Pool
```
* **Why not process-based `multiprocessing.Pool` for Spark?**
  1. **Distributed Compute vs Local Compute**: Heavy transformations run on distributed Spark executors, **not** in Python memory. Python's role is strictly driver orchestration (submitting DAGs to the JVM).
  2. **Py4J Gateway Sockets**: The Py4J gateway maintains a thread-safe connection pool where each Python thread receives a dedicated socket connection to the JVM driver. Forking separate OS processes duplicates file descriptors and can cause socket synchronization collisions.
  3. **Thread-Local Spark Scheduler Pools**: Spark's scheduler properties (such as `sc.setLocalProperty("spark.scheduler.pool", pool_name)` and `sc.setJobGroup(...)`) are stored in Java's `InheritableThreadLocal`. These properties are bound to **threads** within the Spark driver!
  4. **Identical API**: `ThreadPool` shares the exact same `Pool` API (`pool.apply_async(...)`, `pool.map(...)`, `pool.close()`, `pool.join()`), allowing seamless transition from multiprocessing concepts to Spark orchestration.


---
## 3. Spark Under Default FIFO Scheduler: The Head-of-Line Blocking Trap

Let us initialize Apache Spark with its default configuration. By default, `spark.scheduler.mode` is set to **`FIFO`** (First-In, First-Out).

We will configure `local[2]` (2 execution slots/cores), which matches standard Google Colab virtual machines and ensures a clear, controlled environment.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Ensure any existing SparkSession is cleanly terminated
if 'spark_fifo' in locals() and spark_fifo:
    spark_fifo.stop()
if 'spark_fair' in locals() and spark_fair:
    spark_fair.stop()

# Initialize SparkSession with default FIFO scheduler mode
spark_fifo = (
    SparkSession.builder
    .appName("Spark_Scheduler_FIFO_Demo")
    .master("local[2]")  # 2 task execution slots
    .config("spark.scheduler.mode", "FIFO")  # Default Spark behavior
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)

sc_fifo = spark_fifo.sparkContext
sc_fifo.setLogLevel("WARN")

print(" Spark Session (FIFO) Initialized Successfully!")
print(f"  - Master            : {sc_fifo.master}")
print(f"  - Active Sched Mode : {sc_fifo.getConf().get('spark.scheduler.mode', 'FIFO (default)')}")
print(f"  - Default Parallelism: {sc_fifo.defaultParallelism}")


### Designing Two Realistic Spark Workloads:
To clearly observe scheduler behavior, we define two jobs:
1. **Job A (Heavy ETL / Batch Job)**:
   - Partitioned DataFrame with **4 partitions**.
   - Each partition performs a CPU-intensive calculation taking **~1.5 seconds**.
   - With 2 cores, 4 partitions take: $\frac{4}{2} \times 1.5\text{s} = \mathbf{3.0\text{s}}$ total execution time.
2. **Job B (Light Interactive Query / Dashboard Metric)**:
   - DataFrame with **2 partitions**.
   - Each partition performs work taking **~1.0 second**.
   - With 2 cores, 2 partitions take: $\frac{2}{2} \times 1.0\text{s} = \mathbf{1.0\text{s}}$ total execution time.


In [ ]:
print("======================================================================")
print(">>> 2A. SEQUENTIAL SPARK EXECUTION (Baseline)")
print("======================================================================")

def partition_workload(partition_idx, duration_sec):
    """
    Simulates partition processing with deterministic execution time.
    """
    start = time.time()
    # Busy sleep / work simulation
    time.sleep(duration_sec)
    yield (partition_idx, time.time() - start)

def execute_spark_job_A(spark_session, label="Job-A (Heavy Batch)"):
    """
    Heavy Spark Job: 4 partitions x 1.5s duration on 2 cores => ~3.0s total runtime.
    """
    job_start = time.time()
    iso_start = datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"  [{iso_start}] [{label}] DISPATCHED to Spark")

    # Create RDD with 4 partitions
    sc = spark_session.sparkContext
    rdd = sc.parallelize(range(4), numSlices=4)
    # Trigger an action that runs all 4 partitions
    # FIX: Ensure 'it' is an iterator before calling next().
    result = rdd.mapPartitions(lambda it: partition_workload(next(iter(it)), duration_sec=1.5)).collect()

    job_end = time.time()
    iso_end = datetime.now().strftime("%H:%M:%S.%f")[:-3]
    duration = job_end - job_start
    print(f"  [{iso_end}] [{label}] COMPLETED (Latency: {duration:.2f}s)")

    return {
        "job_name": label,
        "start": job_start,
        "end": job_end,
        "duration": duration
    }

def execute_spark_job_B(spark_session, label="Job-B (Light Query)"):
    """
    Light Spark Job: 2 partitions x 1.0s duration on 2 cores => ~1.0s total runtime.
    """
    job_start = time.time()
    iso_start = datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"  [{iso_start}] [{label}] DISPATCHED to Spark")

    sc = spark_session.sparkContext
    rdd = sc.parallelize(range(2), numSlices=2)
    # Trigger an action that runs 2 partitions
    # FIX: Apply the same fix here.
    result = rdd.mapPartitions(lambda it: partition_workload(next(iter(it)), duration_sec=1.0)).collect()

    job_end = time.time()
    iso_end = datetime.now().strftime("%H:%M:%S.%f")[:-3]
    duration = job_end - job_start
    print(f"  [{iso_end}] [{label}] COMPLETED (Latency: {duration:.2f}s)")

    return {
        "job_name": label,
        "start": job_start,
        "end": job_end,
        "duration": duration
    }


seq_spark_start = time.time()

# Run Job A, then Job B
res_spark_seq_A = execute_spark_job_A(spark_fifo)
res_spark_seq_B = execute_spark_job_B(spark_fifo)

seq_spark_total = time.time() - seq_spark_start

print(f"\nSequential Spark Wall-Clock Total: {seq_spark_total:.2f}s")
print(f"  - Job A Duration : {res_spark_seq_A['duration']:.2f}s")
print(f"  - Job B Duration : {res_spark_seq_B['duration']:.2f}s")


### Experiment 2B: Triggering Spark Jobs in Parallel via Python Pool (FIFO Mode)

Now, we wrap both job submissions in Python's `Pool` (`multiprocessing.pool.ThreadPool as Pool`), dispatching both jobs at the exact same instant ($t=0$).

* **Developer's Expectation**: Both jobs will execute simultaneously, and Job B (taking only 1.0s) should finish almost immediately!
* **What Actually Happens**: Let us observe the timestamps below:


In [ ]:
from multiprocessing.pool import ThreadPool as Pool

print("======================================================================")
print(">>> 2B. PARALLEL TRIGGER VIA PYTHON POOL (DEFAULT FIFO SCHEDULER)")
print("======================================================================")

fifo_pool = Pool(processes=2)
fifo_wall_start = time.time()

# Dispatch both jobs concurrently at t=0
async_res_A = fifo_pool.apply_async(execute_spark_job_A, (spark_fifo, "Job-A (Heavy Batch)"))
# Slight 50ms stagger to guarantee deterministic submission order
time.sleep(0.05)
async_res_B = fifo_pool.apply_async(execute_spark_job_B, (spark_fifo, "Job-B (Light Query)"))

fifo_pool.close()
fifo_pool.join()

res_spark_fifo_A = async_res_A.get()
res_spark_fifo_B = async_res_B.get()
fifo_wall_total = time.time() - fifo_wall_start

print(f"\nFIFO Parallel Wall-Clock Total: {fifo_wall_total:.2f}s")
print(f"  - Job A Completion Time: {res_spark_fifo_A['duration']:.2f}s")
print(f"  - Job B Completion Time: {res_spark_fifo_B['duration']:.2f}s  <-- LOOK AT THIS!")


###  Critical Analysis of Experiment 2B:
Notice what happened:
1. Python dispatched both **Job A** and **Job B** at the exact same millisecond ($t=0$).
2. However, **Job B finished at ~4.0s**!
3. Even though Job B only needs **1.0 second** of actual compute time, it was forced to wait **3.0 seconds** in Spark's internal FIFO queue while Job A consumed the 2 available cores!
4. **Conclusion**: Wrapping Spark jobs in Python parallel processing provides **zero concurrency benefit** under the default FIFO scheduler when cluster cores are saturated! Job B suffered from severe **head-of-line blocking**.


---
## 4. Spark Under FAIR Scheduler: Unlocking True Multi-Job Parallelism

To enable multiple jobs to share cluster resources simultaneously, Spark provides the **FAIR Scheduler** (`spark.scheduler.mode = FAIR`).

### How the FAIR Scheduler Works:
* Spark assigns tasks between active jobs in a round-robin "fair sharing" fashion.
* If 2 cores are available, Spark allocates **1 core to Job A** and **1 core to Job B** simultaneously!
* **Job Pools**: Jobs can be assigned to distinct scheduler pools using thread-local properties:
  ```python
  spark.sparkContext.setLocalProperty("spark.scheduler.pool", "pool_name")
  spark.sparkContext.setJobGroup("group_id", "description")
  ```
Because `setLocalProperty` is bound to the Python thread/worker, each worker in the pool can assign its job to an independent pool.


In [ ]:
# Terminate the FIFO session before initializing the FAIR session
spark_fifo.stop()
time.sleep(1)

# Initialize SparkSession with FAIR scheduler mode
spark_fair = (
    SparkSession.builder
    .appName("Spark_Scheduler_FAIR_Demo")
    .master("local[2]")  # 2 task execution slots
    .config("spark.scheduler.mode", "FAIR")  # Enable FAIR scheduler!
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)

sc_fair = spark_fair.sparkContext
sc_fair.setLogLevel("WARN")

print(" Spark Session (FAIR Mode) Initialized Successfully!")
print(f"  - Master            : {sc_fair.master}")
print(f"  - Active Sched Mode : {sc_fair.getConf().get('spark.scheduler.mode')}")
print(f"  - Default Parallelism: {sc_fair.defaultParallelism}")


### Defining FAIR Worker Wrappers with Thread-Local Pools
We will now wrap Job A and Job B so that each worker thread sets its own pool and job group before triggering the Spark action:


In [ ]:
def execute_fair_job_A(spark_session):
    """
    Job A assigned to pool 'batch_pool'.
    """
    sc = spark_session.sparkContext
    # Set thread-local scheduler properties
    sc.setLocalProperty("spark.scheduler.pool", "batch_pool")
    sc.setJobGroup("Job_A_Group", "Heavy Batch ETL Processing")
    
    return execute_spark_job_A(spark_session, label="Job-A (FAIR Batch)")

def execute_fair_job_B(spark_session):
    """
    Job B assigned to pool 'interactive_pool'.
    """
    sc = spark_session.sparkContext
    # Set thread-local scheduler properties
    sc.setLocalProperty("spark.scheduler.pool", "interactive_pool")
    sc.setJobGroup("Job_B_Group", "Light Interactive Metric Query")
    
    return execute_spark_job_B(spark_session, label="Job-B (FAIR Interactive)")


In [ ]:
print("======================================================================")
print(">>> 3. PARALLEL TRIGGER VIA PYTHON POOL (FAIR SCHEDULER)")
print("======================================================================")

fair_pool = Pool(processes=2)
fair_wall_start = time.time()

# Dispatch both jobs simultaneously at t=0
async_fair_A = fair_pool.apply_async(execute_fair_job_A, (spark_fair,))
time.sleep(0.05)  # 50ms stagger
async_fair_B = fair_pool.apply_async(execute_fair_job_B, (spark_fair,))

fair_pool.close()
fair_pool.join()

res_spark_fair_A = async_fair_A.get()
res_spark_fair_B = async_fair_B.get()
fair_wall_total = time.time() - fair_wall_start

print(f"\nFAIR Parallel Wall-Clock Total: {fair_wall_total:.2f}s")
print(f"  - Job A Completion Time: {res_spark_fair_A['duration']:.2f}s")
print(f"  - Job B Completion Time: {res_spark_fair_B['duration']:.2f}s  <-- PROVE CONCURRENCY!")


###  The FAIR Breakthrough:
Look at the results:
* Under **FIFO**: Job B took **~4.0s** to finish because it was queued behind Job A.
* Under **FAIR**: Job B completed in **~1.5s to 2.0s**!
* Spark's FAIR scheduler assigned 1 execution core to Job B immediately, allowing the interactive query to finish with minimal latency while Job A continued processing in parallel!


---
## 5. Benchmarking & Visual Timeline Comparison (Gantt Chart)

Let us consolidate our benchmark measurements and visualize the exact execution lifespans of Job A and Job B across all three paradigms.


In [ ]:
# Create a structured comparative summary table
benchmark_data = [
    {
        "Execution Mode": "1. Sequential (Baseline)",
        "Job A Duration (s)": round(res_spark_seq_A["duration"], 2),
        "Job B Duration (s)": round(res_spark_seq_B["duration"], 2),
        "Job B Blocked Wait (s)": round(res_spark_seq_A["duration"], 2),
        "Total Wall-Clock (s)": round(seq_spark_total, 2),
        "Concurrency Outcome": "Strictly Sequential"
    },
    {
        "Execution Mode": "2. Python Parallel + Spark FIFO",
        "Job A Duration (s)": round(res_spark_fifo_A["duration"], 2),
        "Job B Duration (s)": round(res_spark_fifo_B["duration"], 2),
        "Job B Blocked Wait (s)": round(res_spark_fifo_B["start"] - fifo_wall_start + (res_spark_fifo_A["duration"] - 1.0), 2),
        "Total Wall-Clock (s)": round(fifo_wall_total, 2),
        "Concurrency Outcome": "Head-of-Line Blocking (Pseudo-Parallel)"
    },
    {
        "Execution Mode": "3. Python Parallel + Spark FAIR",
        "Job A Duration (s)": round(res_spark_fair_A["duration"], 2),
        "Job B Duration (s)": round(res_spark_fair_B["duration"], 2),
        "Job B Blocked Wait (s)": 0.0,
        "Total Wall-Clock (s)": round(fair_wall_total, 2),
        "Concurrency Outcome": "True Resource Sharing (Zero Starvation)"
    }
]

df_benchmark = pd.DataFrame(benchmark_data)
display(df_benchmark)


### High-Resolution Gantt Chart: Visualizing Job Lifespans
The plot below clearly illustrates:
1. **Sequential**: Job B cannot start until Job A finishes.
2. **FIFO**: Job B is dispatched at $t=0$, but sits idle waiting for Job A's tasks to release executor cores.
3. **FAIR**: Both Job A and Job B make simultaneous progress from $t=0$, and Job B finishes early!


In [ ]:
# Plot Gantt Chart of Job Executions
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

modes = [
    ("1. Sequential (No Concurrency)", res_spark_seq_A, res_spark_seq_B, seq_spark_start),
    ("2. Python Pool + Spark FIFO (Blocked)", res_spark_fifo_A, res_spark_fifo_B, fifo_wall_start),
    ("3. Python Pool + Spark FAIR (Concurrent)", res_spark_fair_A, res_spark_fair_B, fair_wall_start)
]

colors = {
    "Job A": "#1f77b4",  # Steel Blue
    "Job B": "#ff7f0e",  # Safety Orange
    "Wait": "#d3d3d3"    # Light Gray
}

for idx, (title, job_a, job_b, t_zero) in enumerate(modes):
    ax = axes[idx]
    
    # Relative timing normalized to t_zero
    a_start_rel = max(0.0, job_a["start"] - t_zero)
    a_duration = job_a["duration"]
    
    b_start_rel = max(0.0, job_b["start"] - t_zero)
    b_duration = job_b["duration"]
    
    # Bar for Job A
    ax.barh(y="Job A (Heavy)", width=a_duration, left=a_start_rel, color=colors["Job A"], height=0.45, label="Job A (Active)" if idx == 0 else "")
    
    # For FIFO: indicate the queue wait delay for Job B
    if "FIFO" in title:
        # Job B was waiting during Job A's run
        wait_duration = a_duration - 0.05
        ax.barh(y="Job B (Light)", width=wait_duration, left=0.0, color=colors["Wait"], edgecolor='red', linestyle='--', height=0.45, label="Queue Wait / Starvation" if idx == 1 else "")
        ax.barh(y="Job B (Light)", width=1.0, left=wait_duration, color=colors["Job B"], height=0.45, label="Job B (Active)" if idx == 0 else "")
    elif "Sequential" in title:
        ax.barh(y="Job B (Light)", width=b_duration, left=a_duration, color=colors["Job B"], height=0.45)
    else:
        # FAIR: Job B runs immediately
        ax.barh(y="Job B (Light)", width=b_duration, left=b_start_rel, color=colors["Job B"], height=0.45)

    ax.set_title(title, fontsize=12, fontweight='bold', pad=8, loc='left')
    ax.set_xlim(0, max(seq_spark_total, fifo_wall_total, fair_wall_total) + 0.8)
    ax.grid(axis='x', linestyle='--', alpha=0.7)

axes[-1].set_xlabel("Elapsed Wall-Clock Time (seconds)", fontsize=11, fontweight='bold')
fig.suptitle("Spark Scheduler Comparison: FIFO Head-of-Line Blocking vs. FAIR Multitasking", fontsize=14, fontweight='bold', y=0.98)
fig.legend(loc="upper right", bbox_to_anchor=(0.95, 0.96), ncol=3, frameon=True)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()


---
## 6. Production Architecture & Advanced FAIR Pool Configuration

### When is the FAIR Scheduler Essential?
| Use Case | Scenario & Impact |
| :--- | :--- |
| **Interactive Shared Clusters** | Databricks, Google Colab, or JupyterHub where multiple users run ad-hoc queries concurrently on a single shared Spark application. |
| **Microservices / REST APIs** | An API server querying pre-warmed / cached DataFrames. FAIR guarantees low response latency for short user requests. |
| **Streaming + Batch Hybrids** | Structured Streaming jobs running micro-batches alongside scheduled batch maintenance jobs within the same context. |
| **SLA Enforcement** | Preventing multi-hour ETL pipelines from starving mission-critical compliance reports. |

---

### Advanced Pool Configuration: `fairscheduler.xml`
In production, Spark allows configuring pools with **weights** and **minimum shares (`minShare`)** via an XML configuration file:

```xml
<?xml version="1.0"?>
<allocations>
  <!-- Interactive Pool for low-latency queries -->
  <pool name="interactive_pool">
    <schedulingMode>FAIR</schedulingMode>
    <weight>3</weight>
    <minShare>2</minShare>
  </pool>
  
  <!-- Batch Pool for heavy background ETL -->
  <pool name="batch_pool">
    <schedulingMode>FIFO</schedulingMode>
    <weight>1</weight>
    <minShare>1</minShare>
  </pool>
</allocations>
```

#### Key Pool Attributes:
* **`minShare`**: The minimum number of CPU cores/slots the pool is guaranteed to receive before fair sharing begins.
* **`weight`**: Determines the proportion of cluster resources the pool receives when multiple pools compete for surplus capacity. (A pool with weight 3 gets 3x more slots than a pool with weight 1).
* **`schedulingMode`**: Can be `FIFO` or `FAIR` **within** that specific pool.

To use the allocation file in Spark:
```python
spark = (
    SparkSession.builder
    .config("spark.scheduler.mode", "FAIR")
    .config("spark.scheduler.allocation.file", "/path/to/fairscheduler.xml")
    .getOrCreate()
)
```


In [ ]:
# Step 7: Clean Shutdown of SparkSession
if 'spark_fair' in locals() and spark_fair:
    spark_fair.stop()
    print("SparkSession cleanly stopped.")
